# Python translation of `Final_ML.R`

This notebook reproduces the same workflow as the R script:

1. Read SNP IDs and PGS weights.
2. Identify overlapping variants.
3. Load overlap genotype data and compute PRS.
4. Standardize PRS and simulate binary disease labels.
5. Train Logistic Regression, Random Forest, and SVM models.
6. Evaluate AUROC, Brier score, Accuracy, and draw a calibration plot.

Note: the original R script references `y_test` without defining it. In this notebook, `y_test` is taken correctly from the test split.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [5]:
# Update this if your data files are stored elsewhere.
DATA_DIR = Path('.')

chr9_ids_path = DATA_DIR / 'chr9_ids.txt'
pgs_path = DATA_DIR / 'PGS004197.txt'
chr9_raw_path = DATA_DIR / 'chr9.raw'
chr9_overlap_path = DATA_DIR / 'chr9_overlap.raw'

required_files = [chr9_ids_path, pgs_path, chr9_raw_path, chr9_overlap_path]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        'Missing required files. Put the files next to this notebook or change DATA_DIR:\n'
        + '\n'.join(missing_files)
    )

FileNotFoundError: Missing required files. Put the files next to this notebook or change DATA_DIR:
chr9_ids.txt
PGS004197.txt
chr9.raw
chr9_overlap.raw

In [ ]:
# Read inputs
chr9_ids = pd.read_csv(chr9_ids_path, header=None, names=['rsID'])
pgs = pd.read_csv(pgs_path, sep='\t', comment='#')

# Find overlaps between chr9 IDs and PGS rsIDs
overlap_ids = sorted(set(chr9_ids['rsID']).intersection(pgs['rsID']))
Path('overlap_ids.txt').write_text('\n'.join(overlap_ids) + ('\n' if overlap_ids else ''))

# Read only the header of chr9.raw to locate the columns that should be kept
with chr9_raw_path.open('r', encoding='utf-8') as f:
    header = f.readline().strip().split()

keep_names = {'FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENOTYPE', *overlap_ids}
idx = [i + 1 for i, col in enumerate(header) if col in keep_names]

# Match the R behavior of stripping trailing allele suffixes such as _A
cleaned_header = pd.Series(header).str.replace(r'_[^_]+$', '', regex=True)
Path('keep_cols.txt').write_text(','.join(map(str, idx)) + '\n')

print(f'Header length: {len(header)}')
print(f'Overlap SNP count from ID lists: {len(overlap_ids)}')

In [ ]:
# Load the already-filtered overlap genotype file
geno = pd.read_csv(chr9_overlap_path, sep='\s+')

# Clean SNP column names to remove allele suffixes, keeping the first 6 metadata columns unchanged
geno_columns = list(geno.columns[:6]) + [
    pd.Series(geno.columns[6:]).str.replace(r'_[^_]+$', '', regex=True).tolist()[i]
    for i in range(len(geno.columns[6:]))
]
geno.columns = geno_columns

overlap_ids_geno = [col for col in geno.columns[6:] if col in set(pgs['rsID'])]
pgs_sub = pgs.loc[pgs['rsID'].isin(overlap_ids_geno)].copy()
pgs_sub = pgs_sub.set_index('rsID').loc[overlap_ids_geno].reset_index()

geno_snps = geno[overlap_ids_geno]

print(f'Overlap SNP count in genotype file: {len(overlap_ids_geno)}')
print(f'Rows in pgs_sub: {len(pgs_sub)}')

In [ ]:
# Compute PRS = genotype matrix dot effect weights
prs_scores = geno_snps.to_numpy() @ pgs_sub['effect_weight'].to_numpy()

prs_df = pd.DataFrame({
    'IID': geno['IID'],
    'PRS': prs_scores
})

scaler = StandardScaler()
prs_df['PRS_z'] = scaler.fit_transform(prs_df[['PRS']]).ravel()

# Simulate binary labels using the same logic as the R script
np.random.seed(88)
prob = 1 / (1 + np.exp(-0.8 * prs_df['PRS_z'].to_numpy()))
prs_df['label'] = np.random.binomial(1, prob)

prs_df.to_csv('PRS_dataset.csv', index=False)
prs_df.head()

In [ ]:
# Train/test split
train_df, test_df = train_test_split(
    prs_df,
    test_size=0.3,
    random_state=88,
    shuffle=True
)

X_train = train_df[['PRS_z']]
X_test = test_df[['PRS_z']]
y_train = train_df['label']
y_test = test_df['label']

print(train_df.shape, test_df.shape)

In [ ]:
# Logistic Regression
fit_logit = LogisticRegression(random_state=88)
fit_logit.fit(X_train, y_train)
pred_logit = fit_logit.predict_proba(X_test)[:, 1]

# Random Forest
fit_rf = RandomForestClassifier(n_estimators=300, random_state=88)
fit_rf.fit(X_train, y_train)
pred_rf = fit_rf.predict_proba(X_test)[:, 1]

# SVM with probability output
fit_svm = SVC(kernel='rbf', probability=True, random_state=88)
fit_svm.fit(X_train, y_train)
pred_svm = fit_svm.predict_proba(X_test)[:, 1]

In [ ]:
def eval_model(y_true, y_prob, model_name):
    y_pred = (y_prob >= 0.5).astype(int)
    return pd.DataFrame([
        {
            'Model': model_name,
            'AUROC': roc_auc_score(y_true, y_prob),
            'Brier': brier_score_loss(y_true, y_prob),
            'Accuracy': accuracy_score(y_true, y_pred),
        }
    ])

results_table = pd.concat([
    eval_model(y_test, pred_logit, 'Logistic'),
    eval_model(y_test, pred_rf, 'Random Forest'),
    eval_model(y_test, pred_svm, 'SVM'),
], ignore_index=True)

results_table

In [ ]:
def make_calibration_df(y_true, y_prob, model_name, n_bins=10):
    df = pd.DataFrame({'y': y_true, 'p': y_prob}).copy()

    # qcut may fail if many duplicate probability values exist, so duplicates='drop' is used.
    df['bin'] = pd.qcut(df['p'], q=n_bins, duplicates='drop')
    calib = (
        df.groupby('bin', observed=False)
        .agg(mean_pred=('p', 'mean'), obs_rate=('y', 'mean'))
        .reset_index(drop=True)
    )
    calib['Model'] = model_name
    return calib

calib_all = pd.concat([
    make_calibration_df(y_test, pred_logit, 'Logistic'),
    make_calibration_df(y_test, pred_rf, 'Random Forest'),
    make_calibration_df(y_test, pred_svm, 'SVM'),
], ignore_index=True)

calib_all.head()

In [ ]:
plt.figure(figsize=(8, 6))

for model_name, group in calib_all.groupby('Model'):
    plt.plot(group['mean_pred'], group['obs_rate'], marker='o', label=model_name)

x = np.linspace(0, 1, 100)
plt.plot(x, x, linestyle='--', color='black', label='Perfect calibration')

plt.title('Calibration Comparison')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Event Rate')
plt.legend()
plt.tight_layout()
plt.show()